Model Description:
---

* EfficientNet_B0, i.e the smallest version, image input of 224x224.
* Used Adam optimizer.
* For training, i unfroze Classifier and the last 3 blocks right from the start.
* 100 epochs, training acc rises to ~92% but validation accuracy stayed below 20%
* Test set was at 17% accuracy,



### Functions
---

In [1]:
def printer(iterable:list|dict):
    """
    Displays Lists and Dictionaries nicely
    """
    if isinstance(iterable,list):
        # print("List".center(16,"-"))
        for item in iterable:
            print(f" - {item}")
        print()
    elif isinstance(iterable,dict):
        # print("Dictionary".center(16,"-"))
        
        for key in list(iterable.keys()):
            print(f" {key} :",iterable[key])
        print()
    else:
        print(iterable)


In [2]:
import os

def getClassNames(path = 'processedData/train'):
    folderNames = [folder for folder in os.listdir(path) if os.path.isdir(os.path.join(path, folder))]
    cleanedNames = []
    for idx in range(len(folderNames)):
        words = folderNames[idx].split()
        words.pop(0)
        words.pop()
        if "-" in words:
            words.pop(words.index("-"))

        out = " ".join(words)
        cleanedNames.append(out)
    classMapping = {}
    for index, value in enumerate(cleanedNames):
        classMapping[index] = value
    print(classMapping)

    return classMapping

In [3]:
def giveMeImageSize(modelVersion = "B4") -> tuple:
    """
    Returns the resolution for the respective EfficientNet model verisons:

    "B0", "B1", "B2", "B3", "B4", "B"5, "B6", "B7"

    """
    if modelVersion == "B0":
        return (224,224)
    elif modelVersion == "B1":
        return (240,240)
    elif modelVersion == "B2":
        return (260,260)
    elif modelVersion == "B3":
        return (300,300)
    elif modelVersion == "B4":
        return (380,380)
    elif modelVersion == "B5":
        return (456,456)
    elif modelVersion == "B6":
        return (528,528)
    elif modelVersion == "B7":
        return (600,600)


In [4]:
from torchvision import transforms
#? function for image transformation
def pad_to_Square(image):
    width, height = image.size
    max_side = max(width,height)
    left_pad = (max_side-width)//2
    right_pad = (max_side-width-left_pad)
    top_pad = (max_side-height)//2
    bot_pad = (max_side-height - top_pad)

    padded_image = transforms.functional.pad(image, (left_pad, top_pad, right_pad, bot_pad), padding_mode='constant', fill=0)
    return padded_image


In [5]:
import os
from sklearn.model_selection import train_test_split
from PIL import Image
# Function to create the necessary directories
def create_dirs(base_dir, classes):
    os.makedirs(base_dir, exist_ok=True)
    for subset in ['train', 'valid', 'test']:
        subset_path = os.path.join(base_dir, subset)
        os.makedirs(subset_path, exist_ok=True)
        for class_name in classes:
            os.makedirs(os.path.join(subset_path, class_name), exist_ok=True)

# Function to split and save images into train and test directories
def split_and_process_images(raw_data_dir:str, processed_data_dir:str, randomState=42):
    # Get all class names (subfolder names)
    classes = os.listdir(raw_data_dir)
    print("Subfolders:",classes)
    create_dirs(processed_data_dir, classes)
    
    # Iterate through each class folder
    for class_name in classes:
        class_folder = os.path.join(raw_data_dir, class_name)
        if os.path.isdir(class_folder):
            # Get all image filenames
            image_filenames = os.listdir(class_folder)
            print(f"{class_name} : {len(image_filenames)} files.")
            # Split into train and test sets
            train_files, testval_files = train_test_split(image_filenames, test_size=0.2, random_state=randomState)
            print(f" Train: {len(train_files)}, Test&Val: {len(testval_files)}")
            # Process and save training images
            for filename in train_files:
                image_path = os.path.join(class_folder, filename)
                img = Image.open(image_path)
                # Save the training image
                train_save_path = os.path.join(processed_data_dir, 'train', class_name, filename)
                img.save(train_save_path)

            # Split into validation and test sets
            val_files, test_files = train_test_split(testval_files, test_size=0.5, random_state=randomState)

            # Process and save validation images
            for filename in val_files:
                image_path = os.path.join(class_folder, filename)
                img = Image.open(image_path)
                # Save the validation image
                val_save_path = os.path.join(processed_data_dir, 'valid', class_name, filename)
                img.save(val_save_path)

            # Process and save test images
            for filename in test_files:
                image_path = os.path.join(class_folder, filename)
                img = Image.open(image_path)
                # Save the testing image
                test_save_path = os.path.join(processed_data_dir, 'test', class_name, filename)
                img.save(test_save_path)

        else:
            print(f"Error on {class_name}")


In [6]:
def showGrads(model):
    for name, param in model.named_parameters():
        print(f"Layer: {name}".ljust(46), f"requires_grad: {param.requires_grad}")

def freeze_model(model):
    for params in model.parameters():
        params.requires_grad=False

def unfreeze_model(model):
    for params in model.parameters():
        params.requires_grad = True 
    
def unfreeze_last_n_blocks(model,n):
    lastblock = len(model.features)-1
    blocknames = ["classifier"] + [f"features.{lastblock-i}" for i in range(n)]
    for name,params in model.named_parameters():
        if any(substring in name for substring in blocknames): 
            params.requires_grad = True

    print(f"Layers unfrozen: {blocknames}")


### Data Loaders and Processing
---

#### Cleaning folder names into class names

In [7]:
import torch
from torchvision import datasets, transforms
from torch.utils.data import DataLoader

In [8]:
imgSize = giveMeImageSize("B2")
print(imgSize)

(260, 260)


#### Getting Mean and STD for image normalization
---

In [ ]:
collectMean = False  #? <- Update Mean and STD if model is chagned or image is resized


if collectMean:
    #? Getting the mean and std of the raw dataset
    initial_Transform = transforms.Compose([
        transforms.Lambda(pad_to_Square),
        transforms.Resize(imgSize),  # Resize images
        transforms.ToTensor(),  # Convert images to tensor
    ])

    #? raw dataset File path
    raw_path = './data'

    # Load raw dataset
    dataset = datasets.ImageFolder(root=raw_path, transform=initial_Transform)
    loader = DataLoader(dataset, batch_size=16, shuffle=False)       #? num_workers indicate the number of parallel processes

    # Initialize sums
    mean = 0.
    std = 0.
    total_images = 0

    print("dataset:", dataset)
    batch = 0
    print()
    for images, _ in loader:
        batch_samples = images.size(0)  # batch size
        images = images.view(batch_samples, images.size(1), -1)  # flatten H and W
        mean += images.mean(2).sum(0)
        std += images.std(2).sum(0)
        total_images += batch_samples
        batch += 1
        print(f"Batch [{batch}/{len(loader)}]".ljust(20),
            f"Mean: {(mean/total_images)}",
            f"STD: {(std/total_images)}", end="\r")
    print()

    mean /= total_images
    std /= total_images

    print(f"Mean: {mean}")
    print(f"Std: {std}")

#### Load Datasets:
- Training DataSet with augmentation.
- Testing DataSet without augmentation.
---


In [9]:
 #? Copy form previous cell outputs
# mean = torch.tensor([0.5246, 0.3975, 0.3844])
# std = torch.tensor([0.2752, 0.2196, 0.2174])

#? Pretrained model Mean and SD
imageNet_mean = torch.tensor([0.485, 0.456, 0.406])
imageNet_SD = torch.tensor([0.229, 0.224, 0.225])

transform_train = transforms.Compose([
    transforms.Lambda(pad_to_Square),
    transforms.Resize(imgSize),  # Resize images
    transforms.ToTensor(),  # Convert images to tensor
    transforms.Normalize(mean=imageNet_mean, std=imageNet_SD),  # Normalize images
])

transform_test = transforms.Compose([
    transforms.Lambda(pad_to_Square),
    transforms.Resize(imgSize),  # Resize images
    transforms.ToTensor(),  # Convert images to tensor
])


In [10]:

#? training dataset directory
train_path = "processedData/train"
#? validation dataset directory
valid_path = "processedData/valid"
#? testing dataset directory
test_path = "processedData/test"

batchSize = 64
if train_path:
    #? Load TRAIN dataset from directory
    train_dataset = datasets.ImageFolder(root=train_path, transform=transform_train)
    #? Create a DataLoader
    train_dataloader = DataLoader(train_dataset, batch_size=batchSize, shuffle=True)
    print(train_dataset, f"Batches: {len(train_dataloader)}","\n".ljust(50,"-"))

if valid_path:
    #? Load VALID dataset from directory
    valid_dataset = datasets.ImageFolder(root=valid_path, transform=transform_test)
    #? Create a DataLoader
    valid_dataloader = DataLoader(valid_dataset, batch_size=batchSize, shuffle=False)
    print(valid_dataset, f"Batches: {len(valid_dataloader)}","\n".ljust(50,"-"))

if test_path:
    #? Load TEST dataset from directory
    test_dataset = datasets.ImageFolder(root=test_path, transform=transform_test)
    #? Create a DataLoader
    test_dataloader = DataLoader(test_dataset, batch_size=batchSize, shuffle=False)
    print(test_dataset, f"Batches: {len(test_dataloader)}","\n".ljust(50,"-"))

Dataset ImageFolder
    Number of datapoints: 21719
    Root location: processedData/train
    StandardTransform
Transform: Compose(
               Lambda()
               Resize(size=(260, 260), interpolation=bilinear, max_size=None, antialias=True)
               ToTensor()
               Normalize(mean=tensor([0.4850, 0.4560, 0.4060]), std=tensor([0.2290, 0.2240, 0.2250]))
           ) Batches: 340 
-------------------------------------------------
Dataset ImageFolder
    Number of datapoints: 2715
    Root location: processedData/valid
    StandardTransform
Transform: Compose(
               Lambda()
               Resize(size=(260, 260), interpolation=bilinear, max_size=None, antialias=True)
               ToTensor()
           ) Batches: 43 
-------------------------------------------------
Dataset ImageFolder
    Number of datapoints: 2719
    Root location: processedData/test
    StandardTransform
Transform: Compose(
               Lambda()
               Resize(size=(260, 260)

### Model Training
---

##### Model
---

In [17]:
import torch.nn as nn
import torch.optim as optim
from torchvision import models
import matplotlib.pyplot as plt


efficientnetmodel = models.efficientnet_b2()

no_features = efficientnetmodel.classifier[1].in_features  
efficientnetmodel.classifier[1] = nn.Linear(no_features, 10) 

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
efficientnetmodel = efficientnetmodel.to(device)

criterion = nn.CrossEntropyLoss()
optimiser = optim.Adam(efficientnetmodel.parameters(), lr=0.0001)

freeze_model(efficientnetmodel)
showGrads(efficientnetmodel)

classNames = getClassNames()

Layer: features.0.0.weight                     requires_grad: False
Layer: features.0.1.weight                     requires_grad: False
Layer: features.0.1.bias                       requires_grad: False
Layer: features.1.0.block.0.0.weight           requires_grad: False
Layer: features.1.0.block.0.1.weight           requires_grad: False
Layer: features.1.0.block.0.1.bias             requires_grad: False
Layer: features.1.0.block.1.fc1.weight         requires_grad: False
Layer: features.1.0.block.1.fc1.bias           requires_grad: False
Layer: features.1.0.block.1.fc2.weight         requires_grad: False
Layer: features.1.0.block.1.fc2.bias           requires_grad: False
Layer: features.1.0.block.2.0.weight           requires_grad: False
Layer: features.1.0.block.2.1.weight           requires_grad: False
Layer: features.1.0.block.2.1.bias             requires_grad: False
Layer: features.1.1.block.0.0.weight           requires_grad: False
Layer: features.1.1.block.0.1.weight           r

#### Training from Start
---

In [18]:
# import pickle
from datetime import datetime

epochs = 10
blocks = 2                         #? <-- this determines number of blocks to unfreeze

log_path = "modelLogs"
modelVariation = "B2_Adam"
trngMtd = f"bulkUnfreezing_{blocks}blocks_{epochs}epochs"   #? <-- Change this each time you run a new Model or training type
timestamp = f"{datetime.today().strftime('%Y-%m-%d %H-%M')}"
save_dir = f"{log_path}/{timestamp}"
print(save_dir)


efficientnetmodel.train()
unfreeze_last_n_blocks(efficientnetmodel, blocks)
showImages = False
for epoch in range(epochs):

    running_loss = 0.0
    correct_train = 0
    total_train = 0

    for batch, (inputs, labels) in enumerate(train_dataloader):
        inputs, labels = inputs.to(device), labels.to(device)
        optimiser.zero_grad()
        outputs = efficientnetmodel(inputs).squeeze()
    
        loss = criterion(outputs, labels)
        loss.backward()
        optimiser.step()

        running_loss += loss.item()
        _, predicted = torch.max(outputs,1)
        correct_train += (predicted == labels).sum().item()
        total_train += labels.size(0)

        if not showImages:
            print(f" Epoch [{epoch+1}/{epochs}]".ljust(16),
                    f"Batch: {batch+1}".ljust(12),
                    f"Loss: {loss.item():.4f}".ljust(16),
                    f"Accuracy: {((correct_train / total_train)*100):.2f}%".ljust(20),
                    end="\r")

        if showImages and batch%150==0:
            for img,label,output in zip(inputs,labels,predicted):
                # print(img.shape)
                image_np = img.permute(1, 2, 0).cpu().numpy()
                print("Label:",classNames[label.item()])
                print("Pred:",classNames[output.item()])
                # Display the image using Matplotlib
                plt.imshow(image_np)
                plt.axis('off')  # Hide axis
                plt.show()
        
    
    train_accuracy = (correct_train / total_train)*100
    avg_loss = running_loss / len(train_dataloader)

    print(f" Epoch [{epoch+1}/{epochs}]".ljust(16),
            f"Loss: {avg_loss:.4f}".ljust(12),
            f"Accuracy: {train_accuracy:.2f}%".ljust(40))

    if (epoch+1)%5 == 0:
        os.makedirs(save_dir,exist_ok=True)
        checkpoint_filename = f"{save_dir}/{modelVariation}_{trngMtd}_epoch{epoch+1}.pth"
        torch.save(efficientnetmodel.state_dict(), checkpoint_filename)
        print(f"model saved to {checkpoint_filename}")

            
print(" Evaluation ".center(30,"-"))
efficientnetmodel.eval()
correct = 0
total = 0

with torch.no_grad():
    for vBatch, (inputs, labels) in enumerate(valid_dataloader):
        inputs, labels = inputs.to(device), labels.to(device)
        outputs = efficientnetmodel(inputs).squeeze()
        _, predicted = torch.max(outputs,1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()

        print(f"Batch: {vBatch+1}".ljust(12), f"Accuracy: {((correct/total)*100):.2f}%",end="\r")
        

val_accuracy = (correct / total)*100
print(f" Validation Accuracy: {val_accuracy:.2f}%".ljust(50))


modelLogs/2025-03-30 13-52
Layers unfrozen: ['classifier', 'features.8', 'features.7']
 Epoch [1/10]    Loss: 1.8399 Accuracy: 37.64%                        
 Epoch [2/10]    Loss: 1.4976 Accuracy: 46.61%                        
 Epoch [3/10]    Loss: 1.3982 Accuracy: 49.07%                        
 Epoch [4/10]    Loss: 1.3380 Accuracy: 50.93%                        
 Epoch [5/10]    Loss: 1.2892 Accuracy: 52.10%                        
model saved to modelLogs/2025-03-30 13-52/B2_Adam_bulkUnfreezing_2blocks_10epochs_epoch5.pth
 Epoch [6/10]    Loss: 1.2514 Accuracy: 53.56%                        
 Epoch [7/10]    Loss: 1.2143 Accuracy: 54.29%                        
 Epoch [8/10]    Loss: 1.1909 Accuracy: 55.14%                        
 Epoch [9/10]    Loss: 1.1712 Accuracy: 55.51%                        
 Epoch [10/10]   Loss: 1.1506 Accuracy: 56.51%                        
model saved to modelLogs/2025-03-30 13-52/B2_Adam_bulkUnfreezing_2blocks_10epochs_epoch10.pth
--------- Evalua

#### Load Model From Saver
---

In [ ]:
import torch

modelSaver_path = None
modelVariation = None
checkpoint = torch.load(f"{modelSaver_path}/{modelVariation}_phase4_epoch1.pth",weights_only=False)

for key in list(checkpoint.keys()):
    print(key)

efficientnetmodel.load_state_dict(checkpoint)

#### Train From Saver
---

In [ ]:

epochsPerPhase = 5
resumePhase = 4  #? <- refer to saveFile
phases = 5


for phase in range(resumePhase,phases):
    print(f"Phase: {phase+1}") 
    #? progressive unfreezing of layers
    unfreeze_last_n_blocks(efficientnetmodel, phase)

    efficientnetmodel.train()
    for epoch in range(epochsPerPhase):

        running_loss = 0.0
        correct_train = 0
        total_train = 0

        for batch, (inputs, labels) in enumerate(train_dataloader):
            inputs, labels = inputs.to(device), labels.to(device)
            optimiser.zero_grad()
            outputs = efficientnetmodel(inputs).squeeze()
        
            loss = criterion(outputs, labels)
            loss.backward()
            optimiser.step()

            running_loss += loss.item()
            _, predicted = torch.max(outputs,1)
            correct_train += (predicted == labels).sum().item()
            total_train += labels.size(0)

            print(f" Epoch [{epoch+1}/{epochsPerPhase}]".ljust(16),
                  f"Batch: {batch+1}".ljust(10),
                  f"Accuracy: {((correct_train / total_train)*100):.2f}%".ljust(20),
                  end="\r")
            
        
        train_accuracy = (correct_train / total_train)*100
        avg_loss = running_loss / len(train_dataloader)

        print(f" Epoch [{epoch+1}/{epochsPerPhase}]".ljust(16),
              f"Loss: {avg_loss:.4f}".ljust(10),
              f"Accuracy: {train_accuracy:.2f}%".ljust(30))

        os.makedirs(modelSaver_path,exist_ok=True)
        checkpoint_filename = modelSaver_path+f"/{modelVariation}_phase{phase+1}_epoch{epoch+1}.pth"
        torch.save(efficientnetmodel.state_dict(), checkpoint_filename)
        print(f"model saved to {checkpoint_filename}")

                
    print(" Evaluation ".center(30,"-"))
    efficientnetmodel.eval()
    correct = 0
    total = 0

    with torch.no_grad():
        for vBatch, (inputs, labels) in enumerate(valid_dataloader):
            inputs, labels = inputs.to(device), labels.float().to(device)
            outputs = efficientnetmodel(inputs).squeeze()
            _, predicted = torch.max(outputs,1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()

            print(f"Epoch: {epoch+1}".ljust(10), f"Batch: {vBatch+1}".ljust(12), f"Accuracy: {((correct/total)*100):.2f}%",end="\r")
            

    val_accuracy = (correct / total)*100
    print(f" Validation Accuracy: {val_accuracy:.2f}%".ljust(50))
    print()

#### Test Model
---

In [13]:

# if test_path:
#     #? Load TEST dataset from directory
#     test_dataset = datasets.ImageFolder(root=test_path, transform=transform_test)
#     #? Create a DataLoader
#     test_dataloader = DataLoader(test_dataset, batch_size=4, shuffle=False)
#     print(test_dataset, f"Batches: {len(test_dataloader)}","\n".ljust(50,"-"))

print(" Testing ".center(30,"-"))
efficientnetmodel.eval()
correct = 0
total = 0

with torch.no_grad():
    for tBatch, (inputs, labels) in enumerate(test_dataloader):
  
        inputs, labels = inputs.to(device), labels.to(device)
        outputs = efficientnetmodel(inputs)
        _, predicted = torch.max(outputs,1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()
        
        # if tBatch%50==0:
        #     for img,label,output in zip(inputs,labels,predicted):
        #         # print(img.shape)
        #         image_np = img.permute(1, 2, 0).cpu().numpy()
        #         print(classNames[label.item()])
        #         print(classNames[output.item()])
        #         # Display the image using Matplotlib
        #         plt.imshow(image_np)
        #         plt.axis('off')  # Hide axis
        #         plt.show()

        print(f"Batch: {tBatch+1}".ljust(12), f"Accuracy: {((correct/total)*100):.2f}%",end="\r")
        # break
        

val_accuracy = (correct / total)*100
print(f" Validation Accuracy: {val_accuracy:.2f}%".ljust(50))
print()

---------- Testing -----------
 Validation Accuracy: 12.47%                      

